# ZelusBench: Structural Attention

**How does dependency graph topology affect accuracy?**

This task varies the DAG structure while holding total node count constant.
It tests whether the model can handle branching, merging, and diamond
dependency patterns — not just linear chains.

Topologies:
- **Linear**: A -> B -> C -> D (simple chain)
- **Branching**: A -> B, A -> C (tree structure)
- **Merging**: Two chains merge via midpoint
- **Diamond**: A -> B, A -> C, D = f(B, C) (converging paths)

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import numpy as np
import re
import os

os.environ["RENDER_SUBRUNS"] = "False"

In [ ]:
from zelusbench.scenarios.config import (
    ScenarioConfig, DAGStructure, QueryType,
    DistractorLevel, TransformDensity,
)
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.parser import parse_model_response
from zelusbench.evaluation.scorer import score_query


def score_response(response_text, scenario):
    query_dicts = [q.to_dict() for q in scenario.queries]
    parts = re.split(r'\[Query\s+q_\d+\]', response_text)
    if len(parts) > 1:
        parts = parts[1:]
    if len(parts) != len(query_dicts):
        parts = [response_text] * len(query_dicts)
    return [score_query(qd, parse_model_response(rp, qd)) for qd, rp in zip(query_dicts, parts)]

In [ ]:
DAG_STRUCTURES = {
    "LINEAR": DAGStructure.LINEAR,
    "BRANCHING": DAGStructure.BRANCHING,
    "MERGING": DAGStructure.MERGING,
    "DIAMOND": DAGStructure.DIAMOND,
}

@kbench.task(store_task=False)
def structural_scenario(llm, dag_structure: str, seed: int) -> float:
    """Run one scenario with a given DAG topology."""
    cfg = ScenarioConfig(
        dim=3,
        min_chain_depth=4, max_chain_depth=7,
        dag_structure=DAG_STRUCTURES[dag_structure],
        distractor_level=DistractorLevel.CLEAN,
        transform_density=TransformDensity.STATIC,
        query_types=[QueryType.POSITION, QueryType.DISTANCE],
        num_queries=3, num_splits=3,
        seed=seed,
    )
    gen = ScenarioGenerator(cfg)
    scenario = gen.generate(f"structure_{dag_structure}_s{seed}")

    response = llm.prompt(scenario.prompt)
    scores = score_response(response, scenario)
    avg = float(np.mean([s.score for s in scores]))

    for s in scores:
        kbench.assertions.assert_true(
            s.score > 0,
            expectation=(
                f"{s.query_id} (topology={dag_structure}): "
                f"model should handle {dag_structure.lower()} DAG. Tier={s.tier.name}"
            )
        )
    return avg

In [ ]:
eval_data = pd.DataFrame([
    {"dag_structure": structure, "seed": seed}
    for structure in ["LINEAR", "BRANCHING", "MERGING", "DIAMOND"]
    for seed in range(5)
])

print(f"Evaluation matrix: {len(eval_data)} scenarios")
print(f"Topologies: {eval_data.dag_structure.unique().tolist()}")

In [ ]:
@kbench.task(name="zelusbench_structural_attention")
def zelusbench_structural_attention(llm) -> tuple[float, float]:
    """Structural attention: accuracy by DAG topology.

    Returns (mean_accuracy, std_dev).
    """
    with kbench.client.enable_cache():
        runs = structural_scenario.evaluate(
            llm=[llm],
            evaluation_data=eval_data,
            n_jobs=2,
            timeout=180,
            remove_run_files=True,
            stop_condition=lambda r: len(r) == len(eval_data),
            max_attempts=60,
            retry_delay=10,
        )

    results_df = runs.as_dataframe()
    scores = results_df["result"].dropna().tolist()
    accuracy = float(np.mean(scores)) if scores else 0.0
    std = float(np.std(scores)) if scores else 0.0

    kbench.assertions.assert_true(
        len(scores) > 0,
        expectation="At least some scenarios should produce results"
    )
    return accuracy, std

In [ ]:
run = zelusbench_structural_attention.run(kbench.llm)
run

In [ ]:
%choose zelusbench_structural_attention